In [1]:
from spec2selfies.models.invertion_model import InversionModel
from datasets import load_from_disk
import torch
from transformers import AutoTokenizer
import transformers
from datasets import load_from_disk
import random as rd
from spec2selfies.experiments import experiment_from_args
from spec2selfies.run_args import DataArguments, ModelArguments, TrainingArguments
import numpy as np

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


WANDB_STARTUP_DEBUG 1748613426.2169425 launch
WANDB_STARTUP_DEBUG 1748613426.2186599 wait_ports
WANDB_STARTUP_DEBUG 1748613426.4189844 wait_ports_done
WANDB_STARTUP_DEBUG 1748613426.4192903 launch_done


In [2]:
model_args = ModelArguments(use_lora=False, freeze_strategy = 'encoder_and_decoder')
data_args = DataArguments()
training_args = TrainingArguments(experiment="corrector")

Set num workers to 0


In [3]:
experiment = experiment_from_args(model_args, data_args, training_args)

################
Printing from experiment_from_args
ModelArguments(model_name_or_path='./data/saved_models/models--zjunlp--MolGen-large/snapshots/3c6e8fb91a783853a3782552a85efc4df6f96d0a', embedder_model_name='SELFormer', embedder_gaussian_noise_level=0.0, embedder_torch_dtype='float32', embedding_transform_strategy='repeat', encoder_dropout_disabled=False, decoder_dropout_disabled=False, config_overrides=None, config_name=None, tokenizer_name=None, cache_dir=None, model_revision='main', max_seq_length=1024, torch_dtype='float16', num_repeat_tokens=16, embedder_no_grad=True, use_lora=False, embedder_fake_with_zeros=False, use_frozen_embeddings_as_input=True, use_precomputed_hypotheses=True, store_embedder=False, corrector_ignore_hypothesis_embedding=False, embeddings_from_layer_n=None, freeze_strategy='encoder_and_decoder')
################
################
Printing from __init__  experiment.py
ModelArguments(model_name_or_path='./data/saved_models/models--zjunlp--MolGen-large/snapshot

In [4]:
trainer = experiment.load_trainer()

loading file vocab.json
loading file merges.txt
loading file tokenizer.json
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
loading file chat_template.jinja
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
loading configuration file ./data/saved_models/models--zjunlp--MolGen-large/snapshots/3c6e8fb91a783853a3782552a85efc4df6f96d0a/config.json
Model config BartConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "architectures": [
    "BartForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 12,
  "decoder_start_token_id": 2,
  "dropout": 0.1,
  "enc

################
Printing from config() experiment.py
Type of model_args: <class 'spec2selfies.run_args.ModelArguments'>
Is dataclass: True
Fields: ['model_name_or_path', 'embedder_model_name', 'embedder_gaussian_noise_level', 'embedder_torch_dtype', 'embedding_transform_strategy', 'encoder_dropout_disabled', 'decoder_dropout_disabled', 'config_overrides', 'config_name', 'tokenizer_name', 'cache_dir', 'model_revision', 'max_seq_length', 'torch_dtype', 'num_repeat_tokens', 'embedder_no_grad', 'use_lora', 'embedder_fake_with_zeros', 'use_frozen_embeddings_as_input', 'use_precomputed_hypotheses', 'store_embedder', 'corrector_ignore_hypothesis_embedding', 'embeddings_from_layer_n', 'freeze_strategy']
Vars: {'model_name_or_path': './data/saved_models/models--zjunlp--MolGen-large/snapshots/3c6e8fb91a783853a3782552a85efc4df6f96d0a', 'embedder_model_name': 'SELFormer', 'embedder_gaussian_noise_level': 0.0, 'embedder_torch_dtype': 'float32', 'embedding_transform_strategy': 'repeat', 'encoder_dr

All model checkpoint weights were used when initializing BartForConditionalGeneration.

All the weights of BartForConditionalGeneration were initialized from the model checkpoint at ./data/saved_models/models--zjunlp--MolGen-large/snapshots/3c6e8fb91a783853a3782552a85efc4df6f96d0a.
If your task is similar to the task the model of the checkpoint was trained on, you can already use BartForConditionalGeneration for predictions without further training.
Generation config file not found, using a generation config created from the model config.


05/30/2025 15:57:09 - INFO - spec2selfies.experiments - Loading training dataset ./data/scaffold_split_merged_spectra/arrow/hypothesis_dataset/train
05/30/2025 15:57:09 - INFO - spec2selfies.experiments - Loading eval dataset ./data/scaffold_split_merged_spectra/arrow/hypothesis_dataset/eval


Trainer.tokenizer is now deprecated. You should use `Trainer.processing_class = processing_class` instead.


In [5]:
train_params = trainer.get_num_trainable_parameters()
print (f"Trainable parameters : {round(train_params / 10**6, 3)}M")

Trainable parameters : 404.163M


In [6]:
print("Trainable parameters:")
for name, param in trainer.model.named_parameters():
    if param.requires_grad or "embedding_transform" in name:
        print(name)

Trainable parameters:
encoder_decoder.model.shared.weight
encoder_decoder.model.encoder.embed_positions.weight
encoder_decoder.model.encoder.layers.0.self_attn.k_proj.weight
encoder_decoder.model.encoder.layers.0.self_attn.k_proj.bias
encoder_decoder.model.encoder.layers.0.self_attn.v_proj.weight
encoder_decoder.model.encoder.layers.0.self_attn.v_proj.bias
encoder_decoder.model.encoder.layers.0.self_attn.q_proj.weight
encoder_decoder.model.encoder.layers.0.self_attn.q_proj.bias
encoder_decoder.model.encoder.layers.0.self_attn.out_proj.weight
encoder_decoder.model.encoder.layers.0.self_attn.out_proj.bias
encoder_decoder.model.encoder.layers.0.self_attn_layer_norm.weight
encoder_decoder.model.encoder.layers.0.self_attn_layer_norm.bias
encoder_decoder.model.encoder.layers.0.fc1.weight
encoder_decoder.model.encoder.layers.0.fc1.bias
encoder_decoder.model.encoder.layers.0.fc2.weight
encoder_decoder.model.encoder.layers.0.fc2.bias
encoder_decoder.model.encoder.layers.0.final_layer_norm.weigh

In [6]:
train_dataset = trainer.train_dataset
print (train_dataset)

print (train_dataset[0])

Dataset({
    features: ['spectre', 'input_ids', 'attention_mask', 'labels', 'embedder_input_ids', 'embedder_attention_mask', 'hypothesis_input_ids', 'hypothesis_spectra'],
    num_rows: 117061
})
{'spectre': tensor([0.0008, 0.0008, 0.0008,  ..., 0.0008, 0.0008, 0.0008]), 'input_ids': tensor([  0, 139,  19, 140,  70, 100, 139,  35, 139,  35, 104, 165,  19, 139,
         33, 139,  30,  19, 139,  35, 139,  35, 139,  35,  70,  33, 139,  35,
         70,  60,  60,  33, 139,  30,  33, 139,  30, 139,  35, 139,  35, 104,
        105, 139, 139,  35,  19, 104,  90, 139, 139,  20,  19, 163, 140,  60,
         70, 105,   2]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]), 'labels': tensor([  0, 139,  19, 140,  70, 100, 139,  35, 139,  35, 104, 165,  19, 139,
         33, 139,  30,  19, 139,  35, 139,  35, 139,  35,  70,  33, 

In [7]:
item = torch.tensor([2,   0, 139, 139,  35, 139,  35, 104, 100, 139, 104, 139, 139, 104,
         139, 139, 139, 139,  35,  70, 178,   2])

items = [{
        "labels" : [],
        "input_ids" : item
        }, 
        {
        "labels" : [],
        "input_ids" : item
        }]

collator = experiment.get_collator(trainer.tokenizer)
print (trainer.tokenizer)
print (collator)
collated = collator(items)

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


BartTokenizerFast(name_or_path='./data/saved_models/models--zjunlp--MolGen-large/snapshots/3c6e8fb91a783853a3782552a85efc4df6f96d0a', vocab_size=4, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	4: AddedToken("[CH1-1]", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	5: AddedToken("[=S@@]", rs

In [9]:
print (collated.get('input_ids').shape)
print (collated.get('attention_mask').shape)

torch.Size([2, 1024])
torch.Size([2, 1024])


In [10]:
print(collated.get('input_ids')[0])
print (collated.get('attention_mask')[0])

tensor([  2,   0, 139,  ...,   1,   1,   1])
tensor([1, 1, 1,  ..., 0, 0, 0])


In [3]:
dt = load_from_disk("./data/scaffold_split_merged_spectra/arrow/hypothesis_dataset/train")
print (dt)


Dataset({
    features: ['selfies', 'spectre', 'input_ids', 'attention_mask', 'labels', 'length', 'embedder_input_ids', 'embedder_attention_mask', 'hypothesis_input_ids', 'hypothesis_selfies', 'hypothesis_spectra'],
    num_rows: 117061
})


In [8]:
print(dt[0].get('hypothesis_selfies'))

[C][C][Branch1][C][C][O][C][C][C][N][C][C][C][Branch1][#Branch2][C][C][=C][C][=C][C][=N][Ring1][=Branch1][Branch1][=N][N][C][=Branch1][C][=O][C][=C][N][=N][O][Ring1][Branch1][C][C][Ring2][Ring1][Ring2]


In [9]:
print(dt[1].get('hypothesis_input_ids'))

[2, 0, 139, 139, 139, 139, 139, 139, 139, 139, 139, 33, 139, 30, 20, 139, 104, 139, 139, 139, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [11]:
print(dt[2].get('hypothesis_spectra'))

[0.0005819845828227699, 0.0005882989498786628, 0.000591321149840951, 0.0005921897245571017, 0.0005933729116804898, 0.0005954431253485382, 0.0005968630430288613, 0.0006033880636096001, 0.0006009310600347817, 0.0006072954856790602, 0.0006110845133662224, 0.0006132469861768186, 0.000621959799900651, 0.0006261197268031538, 0.0006351553020067513, 0.0006309811142273247, 0.000638665514998138, 0.0006439171847887337, 0.0006482619792222977, 0.00065369694493711, 0.0006633152370341122, 0.0006718949880450964, 0.0006693661562167108, 0.0006762428092770278, 0.0006804163567721844, 0.0006805919110774994, 0.000761409115511924, 0.000788226374424994, 0.0008565995376557112, 0.0008302237838506699, 0.0008239381131716073, 0.0008439149823971093, 0.0008558523841202259, 0.0008666323847137392, 0.000867870170623064, 0.0008752593421377242, 0.00088964105816558, 0.0008961492567323148, 0.0008930590702220798, 0.0008888337179087102, 0.0008921355474740267, 0.0009161632042378187, 0.0009293548064306378, 0.000948978413362056

In [15]:
print(dt[99].get('hypothesis_input_ids'))

[2, 0, 139, 139, 19, 104, 20, 139, 139, 35, 139, 35, 139, 35, 70, 33, 20, 60, 33, 139, 30, 33, 139, 30, 139, 35, 139, 35, 104, 139, 139, 139, 35, 70, 12, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
